In [ ]:
!jupyter lab --version

4.5.4


In [ ]:
!jupyter --version

Selected Jupyter core packages...
IPython          : 8.14.0
ipykernel        : 6.24.0
ipywidgets       : 8.0.4
jupyter_client   : 8.3.0
jupyter_core     : 5.3.1
jupyter_server   : 2.7.0
jupyterlab       : 4.2.5
nbclient         : 0.8.0
nbconvert        : 7.6.0
nbformat         : 5.9.1
notebook         : 6.5.4
qtconsole        : 5.5.2
traitlets        : 5.9.0


In [ ]:
!pip install --upgrade jupyterlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 26.9 MB/s eta 0:00:00


In [ ]:
!pip install geemap

In [ ]:
import ee
import geemap

### The European Space Agency (ESA) WorldCover 10 m 2021 product provides a global land cover map for 2021 at 10 m resolution based on Sentinel-1 and Sentinel-2 data. The WorldCover product comes with 11 land cover classes. See https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v200#bands for more information.

In [ ]:
import ee
import geemap

# Authenticate and initialize with your specific project ID
ee.Authenticate()
ee.Initialize(project='ee-2621kimanimunga')

Map = geemap.Map()
Map.add_basemap("HYBRID")

esa = ee.ImageCollection("ESA/WorldCover/v200").first()
esa_vis = {"bands": ["Map"]}

Map.addLayer(esa, esa_vis, "ESA Land Cover")
Map.add_legend(title="ESA Land Cover", builtin_legend="ESA_WorldCover")

Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

### Creating charts

In [ ]:
histogram = geemap.image_histogram(
    esa, scale=1000, x_label="Land Cover Type", y_label="Area (km2)"
)
histogram

In [ ]:
df = geemap.image_histogram(esa, scale=1000, return_df=True)
df

,key,value
0,10,6.356948e+07
1,20,1.077804e+07
2,30,4.383527e+07
3,40,1.649686e+07
4,50,9.702980e+05
5,60,2.782152e+07
6,70,8.691136e+06
7,80,3.194526e+07
8,90,4.382204e+06
9,95,1.747340e+05


In [ ]:
esa_labels = list(geemap.builtin_legends["ESA_WorldCover"].keys())
esa_labels

['10 Trees',
 '20 Shrubland',
 '30 Grassland',
 '40 Cropland',
 '50 Built-up',
 '60 Barren / sparse vegetation',
 '70 Snow and ice',
 '80 Open water',
 '90 Herbaceous wetland',
 '95 Mangroves',
 '100 Moss and lichen']

In [ ]:
df["label"] = esa_labels
df

,key,value,label
0,10,6.356948e+07,10 Trees
1,20,1.077804e+07,20 Shrubland
2,30,4.383527e+07,30 Grassland
3,40,1.649686e+07,40 Cropland
4,50,9.702980e+05,50 Built-up
5,60,2.782152e+07,60 Barren / sparse vegetation
6,70,8.691136e+06,70 Snow and ice
7,80,3.194526e+07,80 Open water
8,90,4.382204e+06,90 Herbaceous wetland
9,95,1.747340e+05,95 Mangroves


In [ ]:
round(df["value"].sum() / 1e6, 2)

np.float64(217.06)

In [ ]:
geemap.bar_chart(
    df, x="label", y="value", x_label="Land Cover Type", y_label="Area (km2)"
)

In [ ]:
geemap.pie_chart(df, names="label", values="value", height=500)

### Adding Administrative Boundaries

### Continental Scale

In [ ]:
countries = ee.FeatureCollection(geemap.examples.get_ee_path("countries"))
africa = countries.filter(ee.Filter.eq("CONTINENT", "Africa"))
style = {"fillColor": "00000000"}
Map.addLayer(countries.style(**style), {}, "Countries")
Map.addLayer(africa.style(**style), {}, "Africa")
Map.centerObject(africa)
Map

Map(bottom=16569.0, center=[1.2654126066352944, 16.754414486542828], controls=(WidgetControl(options=['positio…

### Extracting Croplands at continental scale

In [ ]:
africa_cropland = esa.eq(40).clipToCollection(africa).selfMask()
Map.addLayer(africa_cropland, {"palette": ["f096ff"]}, "Cropland")
Map.show_layer(name="ESA Land Cover", show=False)

### Continent Scale Zonal Statistics

In [ ]:
geemap.zonal_stats(africa_cropland, africa, "esa_africa_cropland.csv", stat_type="SUM", scale=1000)

Computing statistics ...
Generating URL ...
Please wait ...
Data downloaded to /home/jovyan/esa_africa_cropland.csv


### Country Scale

In [ ]:
countries = ee.FeatureCollection(geemap.examples.get_ee_path("countries"))
kenya = countries.filter(ee.Filter.eq("NAME", "Kenya"))
style = {"fillColor": "00000000"}
Map.addLayer(kenya.style(**style), {}, "Kenya")
Map.centerObject(kenya)
Map

Map(bottom=8476.0, center=[0.3515602939922709, 37.79296875000001], controls=(WidgetControl(options=['position'…

### Extracting Croplands at country scale

In [ ]:
ke_cropland = esa.eq(40).clipToCollection(kenya).selfMask()
Map.addLayer(ke_cropland, {"palette": ["f096ff"]}, "Cropland")
Map.show_layer(name="Kenya ESA Land Cover", show=False)

### Country Scale Zonal Statistics

In [ ]:
geemap.zonal_stats(ke_cropland, kenya, "esa_kenya_cropland.csv", stat_type="SUM", scale=1000)

Computing statistics ...
Generating URL ...
Please wait ...
Data downloaded to /home/jovyan/esa_kenya_cropland.csv


In [ ]:
df = geemap.csv_to_df("esa_kenya_cropland.csv")
df.head()

,sum,GDP_MD_EST,ISO_A2,POP_RANK,ISO_A3,CONTINENT,POP_EST,INCOME_GRP,SUBREGION,system:index,NAME
0,25668.098039,152700.0,KE,15,KEN,Africa,47615739,5. Low income,Eastern Africa,0000000000000000000d,Kenya


### ESRI GLobal Land Cover

The ESRI GLobal Land Cover dataset is a global map of land use/land cover (LULC) derived from ESA Sentinel-2 imagery at 10m resolution. Each year is generated from Impact Observatory’s deep learning AI land classification model used a massive training dataset of billions of human-labeled image pixels developed by the National Geographic Society. The global maps were produced by applying this model to the Sentinel-2 scene collection on Microsoft’s Planetary Computer, processing over 400,000 Earth observations per year. The datasets are available for 2017 ~ 2023.

- https://livingatlas.arcgis.com/landcover/
- https://www.arcgis.com/home/item.html?id=d3da5dd386d140cf93fc9ecbf8da5e31
- https://samapriya.github.io/awesome-gee-community-datasets/projects/S2TSLULC/

In [ ]:
Map = geemap.Map()
Map.add_basemap("HYBRID")

esri = ee.ImageCollection(
    "projects/sat-io/open-datasets/landcover/ESRI_Global-LULC_10m_TS"
)

esri_2017 = esri.filterDate("2017-01-01", "2017-12-31").mosaic()
esri_2018 = esri.filterDate("2018-01-01", "2018-12-31").mosaic()
esri_2019 = esri.filterDate("2019-01-01", "2019-12-31").mosaic()
esri_2020 = esri.filterDate("2020-01-01", "2020-12-31").mosaic()
esri_2021 = esri.filterDate("2021-01-01", "2021-12-31").mosaic()
esri_2022 = esri.filterDate("2022-01-01", "2022-12-31").mosaic()
esri_2023 = esri.filterDate("2023-01-01", "2023-12-31").mosaic()

esri_vis = {"min": 1, "max": 11, "palette": "esri_lulc"}

Map.addLayer(esri_2017, esri_vis, "ESRI LULC 2017")
Map.addLayer(esri_2018, esri_vis, "ESRI LULC 2018")
Map.addLayer(esri_2019, esri_vis, "ESRI LULC 2019")
Map.addLayer(esri_2020, esri_vis, "ESRI LULC 2020")
Map.addLayer(esri_2021, esri_vis, "ESRI LULC 2021")
Map.addLayer(esri_2022, esri_vis, "ESRI LULC 2022")
Map.addLayer(esri_2023, esri_vis, "ESRI LULC 2023")

Map.add_legend(title="ESRI Land Cover", builtin_legend="ESRI_LandCover_TS")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

### Inspecting the Timeseries

In [ ]:
images = ee.List([esri_2017, esri_2018, esri_2019, esri_2020, esri_2021, esri_2022, esri_2023])
collection = ee.ImageCollection.fromImages(images)

In [ ]:
years = [str(year) for year in range(2017, 2024)]
years

['2017', '2018', '2019', '2020', '2021', '2022', '2023']

In [ ]:
Map = geemap.Map()
Map.ts_inspector(collection, years, esri_vis, width="80px")
Map.add_legend(title="ESRI Land Cover", builtin_legend="ESRI_LandCover_TS")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Dropdown(layout=Layo…

### Extracting Croplands

In [ ]:
countries = ee.FeatureCollection(geemap.examples.get_ee_path("countries"))
kenya = countries.filter(ee.Filter.eq("NAME", "Kenya"))

In [ ]:
ke_cropland_col = collection.map(lambda img: img.eq(5).clipToCollection(kenya).selfMask())
ke_cropland_ts = ke_cropland_col.toBands().rename(years)

In [ ]:
Map = geemap.Map()

style = {"fillColor": "00000000"}
Map.addLayer(ke_cropland_col.first(), {"palette": ["#ab6c28"]}, "first")
Map.addLayer(countries.style(**style), {}, "Countries", False)
Map.addLayer(kenya.style(**style), {}, "Kenya")
Map.centerObject(kenya)

Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [ ]:
ke_cropland_ts.bandNames().getInfo()

['2017', '2018', '2019', '2020', '2021', '2022', '2023']

In [ ]:
geemap.zonal_stats(
    ke_cropland_ts, kenya, "esri_kenya_cropland.csv", stat_type="SUM", scale=1000
)

Computing statistics ...
Generating URL ...
Please wait ...
Data downloaded to /home/jovyan/esri_kenya_cropland.csv


### Analyzing Cropland Gain and Loss

In [ ]:
Map = geemap.Map()
Map.add_basemap("HYBRID")

cropland_2017 = esri_2017.eq(5).selfMask()
cropland_2021 = esri_2021.eq(5).selfMask()

cropland_gain = esri_2017.neq(5).And(esri_2021.eq(5)).selfMask()
cropland_loss = esri_2017.eq(5).And(esri_2021.neq(5)).selfMask()

Map.addLayer(cropland_2017, {"palette": "brown"}, "Cropland 2017", False)
Map.addLayer(cropland_2021, {"palette": "cyan"}, "Cropland 2021", False)

Map.addLayer(cropland_gain, {"palette": "yellow"}, "Cropland gain")
Map.addLayer(cropland_loss, {"palette": "red"}, "Cropland loss")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

## Dynamic World Land Cover

Dynamic World is a near realtime 10m resolution global land use land cover dataset, produced using deep learning, freely available and openly licensed. As a result of leveraging a novel deep learning approach, based on Sentinel-2 Top of Atmosphere, Dynamic World offers global land cover updating every 2-5 days depending on location.

- [Dynamic World Website](https://www.dynamicworld.app/)
- [Dynamic World datasets on Earth Engine](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_DYNAMICWORLD_V1)

### Classification and Probability

In [ ]:
Map = geemap.Map()

region = ee.Geometry.BBox(-179, -89, 179, 89)
start_date = "2022-01-01"
end_date = "2023-01-01"

dw_class = geemap.dynamic_world(region, start_date, end_date, return_type="class")
dw = geemap.dynamic_world(region, start_date, end_date, return_type="hillshade")

dw_vis = {"min": 0, "max": 8, "palette": "dw"}

Map.addLayer(dw_class, dw_vis, "DW Land Cover", False)
Map.addLayer(dw, {}, "DW Land Cover Hillshade")

Map.add_legend(title="Dynamic World Land Cover", builtin_legend="Dynamic_World")
Map.setCenter(37.3803, -0.653, 9)
Map

Map(center=[-0.653, 37.3803], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDat…

### ESRI Land Cover vs. Dynamic World

In [ ]:
Map = geemap.Map(center=[37.3803, -0.653], zoom=9)

left_layer = geemap.ee_tile_layer(esri_2022, esri_vis, "ESRI Land Cover")
right_layer = geemap.ee_tile_layer(dw, {}, "Dynamic World Land Cover")

Map.split_map(left_layer, right_layer)
Map.add_legend(
    title="ESRI Land Cover", builtin_legend="ESRI_LandCover", position="bottomleft"
)
Map.add_legend(
    title="Dynamic World Land Cover",
    builtin_legend="Dynamic_World",
    position="bottomright",
)
Map.setCenter(37.3803, -0.653, 9)

Map

Map(center=[-0.653, 37.3803], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

### Visualizing a Region/ Area of Interest (ROI/ AOI)

In [ ]:
Map2 = geemap.Map()
Map2.add_basemap("HYBRID")
Map2

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [ ]:
# Set the region of interest by simply drawing a polygon on the map
region = Map2.user_roi
if region is None:
    region = ee.Geometry.BBox(37.238846,-0.801289,37.238846,-0.504678)
Map2.centerObject(region)

In [ ]:
# Set the date range
start_date = "2024-01-01"
end_date = "2024-08-27"

In [ ]:
# Create a Sentinel-2 image composite
image = geemap.dynamic_world_s2(region, start_date, end_date)
vis_params = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}
Map2.addLayer(image, vis_params, "Sentinel-2 image")

In [ ]:
# Create Dynamic World land cover composite
landcover = geemap.dynamic_world(region, start_date, end_date, return_type="hillshade")
Map2.addLayer(landcover, {}, "Land Cover")

In [ ]:
# Add legend to the map
Map2.add_legend(title="Dynamic World Land Cover", builtin_legend="Dynamic_World")
Map2

Map(bottom=66076.0, center=[-0.6591651462894504, 37.84240683101039], controls=(WidgetControl(options=['positio…